<a href="https://colab.research.google.com/github/Viswas-Vinayakumar/AI-search/blob/main/notebooks/coach_search_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Coach search: keyword vs. semantic vs. hybrid, measured

A small prototype of the problem a coach-matching platform has: a company describes what it needs in plain German, and the system must find the right coach profiles.

**What this notebook does**
1. 30 **synthetic** coach profiles (invented names and bios, not real people).
2. Three ways to search them: **keyword (BM25)**, **semantic (multilingual embeddings)** and **hybrid (Reciprocal Rank Fusion)**, plus hard filters for city and price.
3. A **golden set** of 15 realistic queries with hand-labelled correct coaches, scored with **Recall@5** and **MRR@10**, then an error analysis of where each method fails.

**Run it:** Runtime → Run all. No GPU needed (about 2 minutes).

In [ ]:
# 1. Install
!pip -q install rank_bm25 sentence-transformers

In [ ]:
# 2. The data: 30 synthetic coach profiles (invented for this demo)
PROFILES = [
 {"id": "c01", "name": "Anna B.",     "city": "Hamburg",    "price": 1800, "title": "Konfliktcoach für Teams",
  "bio": "Ich begleite Teams, in denen es knirscht: Spannungen, schwierige Gespräche und festgefahrene Konflikte. Schwerpunkt Vertriebs- und Serviceteams. Methoden: Mediation und gewaltfreie Kommunikation."},
 {"id": "c02", "name": "Markus H.",   "city": "Berlin",     "price": 2200, "title": "Führungskräftetraining",
  "bio": "Programme für neue Führungskräfte: der Rollenwechsel vom Kollegen zur Führungskraft, Delegation, Verantwortung übernehmen."},
 {"id": "c03", "name": "Sophie K.",   "city": "München",    "price": 1500, "title": "Agile Coach und Scrum-Trainerin",
  "bio": "Zertifizierte Scrum-Trainerin (PSM, PSPO). Einführung von Scrum und Kanban in Softwareteams, Retrospektiven moderieren."},
 {"id": "c04", "name": "Jonas W.",    "city": "Köln",       "price": 1200, "title": "Präsentation und Rhetorik",
  "bio": "Souverän vor Publikum: Stimme, Körpersprache, Lampenfieber. Für Vorträge, Pitches und Kundenpräsentationen."},
 {"id": "c05", "name": "Lena S.",     "city": "Online",     "price": 900,  "title": "Remote-Teams und hybride Zusammenarbeit",
  "bio": "Wie verteilte Teams gut zusammenarbeiten: virtuelle Meetings, Vertrauen auf Distanz, Rituale für Teams im Homeoffice."},
 {"id": "c06", "name": "Thomas B.",   "city": "Frankfurt",  "price": 2500, "title": "Verhandlungstraining",
  "bio": "Das Harvard-Konzept in der Praxis: Preisverhandlungen im B2B-Vertrieb, Einkauf und Lieferantengespräche."},
 {"id": "c07", "name": "Miriam W.",   "city": "Hamburg",    "price": 1400, "title": "Resilienz und Stressmanagement",
  "bio": "Burnout vorbeugen, mit Druck umgehen, eine gesunde Pausenkultur. Workshops zur mentalen Gesundheit am Arbeitsplatz."},
 {"id": "c08", "name": "David F.",    "city": "Berlin",     "price": 1600, "title": "Change Management",
  "bio": "Begleitung von Veränderungsprozessen: Umstrukturierung, Einführung neuer Software, Fusionen. Widerstände verstehen und Mitarbeitende mitnehmen."},
 {"id": "c09", "name": "Julia N.",    "city": "München",    "price": 1300, "title": "Zeitmanagement und Selbstorganisation",
  "bio": "Prioritäten setzen, Eisenhower-Matrix, die E-Mail-Flut bändigen. Für Fachkräfte mit vollem Kalender."},
 {"id": "c10", "name": "Kerstin B.",  "city": "Hamburg",    "price": 2000, "title": "Vertriebstraining",
  "bio": "Neukundengewinnung, Kaltakquise am Telefon, Abschlusstechniken. Trainings für Außendienst und Innendienst."},
 {"id": "c11", "name": "Felix Z.",    "city": "Online",     "price": 1100, "title": "KI im Arbeitsalltag",
  "bio": "Praxisworkshops zu ChatGPT, Copilot und Co.: Prompts schreiben, Datenschutz beachten, Routinearbeit automatisieren."},
 {"id": "c12", "name": "Sandra K.",   "city": "Köln",       "price": 1700, "title": "Teamentwicklung",
  "bio": "Teambuilding mit Tiefgang: Rollen klären, Zusammenarbeit verbessern, Teamworkshops mit erlebnisorientierten Methoden."},
 {"id": "c13", "name": "Martin H.",   "city": "Frankfurt",  "price": 1900, "title": "Feedback- und Gesprächskultur",
  "bio": "Konstruktives Feedback geben und annehmen, Kritikgespräche ohne Eskalation, eine offene Gesprächskultur im Team."},
 {"id": "c14", "name": "Aylin D.",    "city": "Berlin",     "price": 1500, "title": "Interkulturelle Kompetenz und Diversity",
  "bio": "Zusammenarbeit in internationalen Teams, unbewusste Vorurteile (Unconscious Bias), inklusive Führung."},
 {"id": "c15", "name": "Peter S.",    "city": "Stuttgart",  "price": 1400, "title": "DISG-Training",
  "bio": "Persönlichkeitsprofile mit dem DISG-Modell: sich selbst und Kollegen besser verstehen, Kommunikation an den Typ anpassen."},
 {"id": "c16", "name": "Nina L.",     "city": "Hamburg",    "price": 1250, "title": "Moderation und Workshop-Design",
  "bio": "Workshops und Strategietage moderieren, Großgruppenformate, Liberating Structures, Ergebnisse sichern."},
 {"id": "c17", "name": "Oliver S.",   "city": "München",    "price": 2800, "title": "Executive Coaching",
  "bio": "Einzelcoaching für Geschäftsführung und Vorstände: Entscheidungen unter Unsicherheit, Rollenklarheit, Sparring auf Augenhöhe."},
 {"id": "c18", "name": "Hannah W.",   "city": "Online",     "price": 800,  "title": "Onboarding neuer Mitarbeitender",
  "bio": "Strukturierte Einarbeitung, Patenprogramme, die ersten 100 Tage, besonders beim Remote-Onboarding."},
 {"id": "c19", "name": "Stefan K.",   "city": "Köln",       "price": 1600, "title": "Projektmanagement",
  "bio": "Klassisches Projektmanagement nach PMI und PRINCE2: Planung, Risiken, Stakeholder. Vorbereitung auf die PMP-Zertifizierung."},
 {"id": "c20", "name": "Laura R.",    "city": "Berlin",     "price": 1350, "title": "Kundenservice und Beschwerdemanagement",
  "bio": "Freundlich bleiben, wenn Kunden wütend sind: Deeskalation am Telefon und im Chat, Beschwerden als Chance."},
 {"id": "c21", "name": "Ben S.",      "city": "Hamburg",    "price": 1700, "title": "Agile Leadership",
  "bio": "Führen in agilen Organisationen: OKRs, selbstorganisierte Teams, Servant Leadership. Ergänzt Scrum-Einführungen."},
 {"id": "c22", "name": "Claudia M.",  "city": "Frankfurt",  "price": 1450, "title": "Betriebliches Gesundheitsmanagement",
  "bio": "Ergonomie, Bewegung im Büro, gesunde Ernährung. Gesundheitstage und Programme für Unternehmen."},
 {"id": "c23", "name": "Tim H.",      "city": "Berlin",     "price": 1000, "title": "Excel und Datenanalyse",
  "bio": "Excel für Controlling und Vertrieb: Pivot-Tabellen, SVERWEIS, Power Query, Dashboards."},
 {"id": "c24", "name": "Eva S.",      "city": "München",    "price": 1550, "title": "Karriere- und Laufbahncoaching",
  "bio": "Berufliche Neuorientierung, Bewerbungsstrategie und Gehaltsverhandlung für Fachkräfte."},
 {"id": "c25", "name": "Robert L.",   "city": "Hamburg",    "price": 2100, "title": "Mediation bei Konflikten",
  "bio": "Zertifizierter Mediator. Klärung eskalierter Konflikte zwischen Abteilungen oder Geschäftspartnern, vertraulich und neutral."},
 {"id": "c26", "name": "Katrin V.",   "city": "Online",     "price": 950,  "title": "Virtuelle Präsentationen",
  "bio": "Online überzeugen: Präsentieren in Zoom und Teams, Kamera, Licht, Interaktion mit dem digitalen Publikum."},
 {"id": "c27", "name": "Andreas F.",  "city": "Düsseldorf", "price": 1650, "title": "Datenschutz-Schulung (DSGVO)",
  "bio": "DSGVO-Grundlagen für Mitarbeitende, Umgang mit personenbezogenen Daten, Pflichtschulungen mit Zertifikat."},
 {"id": "c28", "name": "Melanie B.",  "city": "Berlin",     "price": 1750, "title": "Teamcoaching für verteilte Teams",
  "bio": "Teamcoaching für Teams, die über mehrere Standorte und Zeitzonen arbeiten: Kommunikationsregeln, Vertrauen, Konflikte auf Distanz."},
 {"id": "c29", "name": "Jan K.",      "city": "Hamburg",    "price": 1300, "title": "Präsentationstraining für den Vertrieb",
  "bio": "Angebote überzeugend präsentieren, Storytelling für Verkaufsgespräche, Pitch-Training für Sales-Teams."},
 {"id": "c30", "name": "Ute P.",      "city": "Köln",       "price": 1100, "title": "Mitarbeitergespräche führen",
  "bio": "Jahresgespräche, Zielvereinbarungen und schwierige Gespräche mit Mitarbeitenden, praxisnah mit Rollenspielen."},
]
by_id = {p["id"]: p for p in PROFILES}
ids = [p["id"] for p in PROFILES]
print(len(PROFILES), "profiles")

30 profiles


In [ ]:
# 3. The golden set: realistic queries + the coaches a human would call correct
# Labels are my own judgement. Each query tests something specific (see "tests").
GOLDEN = [
 {"query": "Konfliktmanagement für unser Vertriebsteam",        "relevant": ["c01", "c25"], "tests": "synonym: no profile says 'Konfliktmanagement'"},
 {"query": "Scrum Einführung",                                    "relevant": ["c03", "c21"], "tests": "exact technical term"},
 {"query": "DISG Persönlichkeitstest",                            "relevant": ["c15"],        "tests": "exact brand name"},
 {"query": "Teamcoaching für Remote-Teams",                       "relevant": ["c05", "c28"], "tests": "two good matches"},
 {"query": "Mitarbeiter haben Angst vor der neuen Software",      "relevant": ["c08"],        "tests": "problem described, not the topic"},
 {"query": "Wie bleibe ich ruhig, wenn ein Kunde schreit?",       "relevant": ["c20"],        "tests": "question style, no shared words"},
 {"query": "Burnout Prävention",                                  "relevant": ["c07"],        "tests": "keyword + paraphrase"},
 {"query": "sales training cold calling",                         "relevant": ["c10"],        "tests": "English query, German profiles"},
 {"query": "Präsentationstraining", "filters": {"city": "Hamburg"}, "relevant": ["c29"],      "tests": "city filter"},
 {"query": "neue Führungskraft: vom Kollegen zum Chef",           "relevant": ["c02"],        "tests": "paraphrase"},
 {"query": "ChatGPT Workshop für Mitarbeitende",                  "relevant": ["c11"],        "tests": "exact product name"},
 {"query": "Gehalt verhandeln",                                   "relevant": ["c24"],        "tests": "German compound word + distractor (c06)"},
 {"query": "Einarbeitung neuer Kollegen im Homeoffice", "filters": {"max_price": 1000}, "relevant": ["c18"], "tests": "paraphrase + price filter"},
 {"query": "Datenschutz Schulung für das ganze Team",             "relevant": ["c27"],        "tests": "keyword with a distractor (c11)"},
 {"query": "Strategieworkshop moderieren", "filters": {"city": "Hamburg"}, "relevant": ["c16"], "tests": "compound word + city filter"},
]
for g in GOLDEN:
    print(f"{g['query']:<52} -> {g['relevant']}  ({g['tests']})")

Konfliktmanagement für unser Vertriebsteam           -> ['c01', 'c25']  (synonym: no profile says 'Konfliktmanagement')
Scrum Einführung                                     -> ['c03', 'c21']  (exact technical term)
DISG Persönlichkeitstest                             -> ['c15']  (exact brand name)
Teamcoaching für Remote-Teams                        -> ['c05', 'c28']  (two good matches)
Mitarbeiter haben Angst vor der neuen Software       -> ['c08']  (problem described, not the topic)
Wie bleibe ich ruhig, wenn ein Kunde schreit?        -> ['c20']  (question style, no shared words)
Burnout Prävention                                   -> ['c07']  (keyword + paraphrase)
sales training cold calling                          -> ['c10']  (English query, German profiles)
Präsentationstraining                                -> ['c29']  (city filter)
neue Führungskraft: vom Kollegen zum Chef            -> ['c02']  (paraphrase)
ChatGPT Workshop für Mitarbeitende                   -> ['c11']  (ex

In [ ]:
# 4. Shared helpers: the text we search, and hard filters
import re, numpy as np, pandas as pd

def doc_text(p):
    return f"{p['title']}. {p['bio']}"

def allowed(p, filters):
    """Hard rules, never left to the model: a coach outside the city or budget is simply removed."""
    if "city" in filters and p["city"] != filters["city"]:
        return False
    if "max_price" in filters and p["price"] > filters["max_price"]:
        return False
    return True

In [ ]:
# 5. Method 1: keyword search (BM25)
from rank_bm25 import BM25Okapi

STOPWORDS = set()       # empty for now; see the experiment at the end

def tokenize(text):
    words = re.findall(r"\w+", text.lower())    # lowercase words, punctuation removed
    return [w for w in words if w not in STOPWORDS]

bm25 = BM25Okapi([tokenize(doc_text(p)) for p in PROFILES])

def keyword_rank(query, filters={}):
    scores = bm25.get_scores(tokenize(query))
    ranked = sorted(zip(ids, scores), key=lambda x: -x[1])
    # score 0 = not a single shared word = not found
    return [(i, float(s)) for i, s in ranked if s > 0 and allowed(by_id[i], filters)]

# See the German compound-word problem with your own eyes:
print(tokenize("Gehalt verhandeln"))
print(tokenize(by_id["c24"]["bio"]))
print("-> 'gehalt' and 'verhandeln' never equal 'gehaltsverhandlung', so BM25 finds nothing.")
print(keyword_rank("Gehalt verhandeln"))

['gehalt', 'verhandeln']
['berufliche', 'neuorientierung', 'bewerbungsstrategie', 'und', 'gehaltsverhandlung', 'für', 'fachkräfte']
-> 'gehalt' and 'verhandeln' never equal 'gehaltsverhandlung', so BM25 finds nothing.
[]


In [ ]:
# 6. Method 2: semantic search (multilingual embeddings)
from sentence_transformers import SentenceTransformer

emb_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
doc_emb = emb_model.encode([doc_text(p) for p in PROFILES], normalize_embeddings=True)
print("each profile is now a vector of", doc_emb.shape[1], "numbers")

def vector_rank(query, filters={}):
    q = emb_model.encode([query], normalize_embeddings=True)[0]
    scores = doc_emb @ q                       # cosine similarity (vectors are normalised)
    ranked = sorted(zip(ids, scores), key=lambda x: -x[1])
    return [(i, float(s)) for i, s in ranked if allowed(by_id[i], filters)]

for i, s in vector_rank("Gehalt verhandeln")[:3]:
    print(f"{s:.3f}  {i}  {by_id[i]['title']}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

each profile is now a vector of 384 numbers
0.425  c30  Mitarbeitergespräche führen
0.423  c25  Mediation bei Konflikten
0.423  c06  Verhandlungstraining


In [ ]:
# 7. Method 3: hybrid, Reciprocal Rank Fusion (RRF)
def hybrid_rank(query, filters={}, k=60):
    """Each list gives every coach 1 / (k + rank). Add them up. A coach high in BOTH lists wins."""
    fused = {}
    for ranking in (keyword_rank(query, filters), vector_rank(query, filters)):
        for rank, (i, _) in enumerate(ranking, start=1):
            fused[i] = fused.get(i, 0.0) + 1.0 / (k + rank)
    return sorted(fused.items(), key=lambda x: -x[1])

METHODS = {"keyword": keyword_rank, "vector": vector_rank, "hybrid": hybrid_rank}

def search(query, filters={}, top=5):
    print(f"QUERY: {query}   FILTERS: {filters or 'none'}")
    for name, fn in METHODS.items():
        top_hits = fn(query, filters)[:top]
        print(f"  {name:<8}", " | ".join(f"{i} {by_id[i]['title'][:28]}" for i, _ in top_hits) or "(nothing found)")

search("Konfliktmanagement für unser Vertriebsteam")
print()
search("Präsentationstraining", {"city": "Hamburg"})

QUERY: Konfliktmanagement für unser Vertriebsteam   FILTERS: none
  keyword  c29 Präsentationstraining für de | c28 Teamcoaching für verteilte T | c24 Karriere- und Laufbahncoachi | c10 Vertriebstraining | c22 Betriebliches Gesundheitsman
  vector   c25 Mediation bei Konflikten | c01 Konfliktcoach für Teams | c20 Kundenservice und Beschwerde | c08 Change Management | c30 Mitarbeitergespräche führen
  hybrid   c28 Teamcoaching für verteilte T | c01 Konfliktcoach für Teams | c02 Führungskräftetraining | c27 Datenschutz-Schulung (DSGVO) | c09 Zeitmanagement und Selbstorg

QUERY: Präsentationstraining   FILTERS: {'city': 'Hamburg'}
  keyword  c29 Präsentationstraining für de
  vector   c29 Präsentationstraining für de | c10 Vertriebstraining | c16 Moderation und Workshop-Desi | c01 Konfliktcoach für Teams | c25 Mediation bei Konflikten
  hybrid   c29 Präsentationstraining für de | c10 Vertriebstraining | c16 Moderation und Workshop-Desi | c01 Konfliktcoach für Teams | c25 Mediation bei Kon

In [ ]:
# 8. Evaluate every method on the golden set
def first_relevant_rank(ranked_ids, relevant, cutoff=10):
    for r, i in enumerate(ranked_ids[:cutoff], start=1):
        if i in relevant:
            return r
    return None                                  # not in the top 10

def evaluate():
    rows = []
    for g in GOLDEN:
        row = {"query": g["query"]}
        for name, fn in METHODS.items():
            ranked = [i for i, _ in fn(g["query"], g.get("filters", {}))]
            hits_in_top5 = len(set(ranked[:5]) & set(g["relevant"]))
            r = first_relevant_rank(ranked, g["relevant"])
            row[f"{name}_recall5"] = hits_in_top5 / len(g["relevant"])
            row[f"{name}_rr"] = 1 / r if r else 0.0      # reciprocal rank
            row[f"{name}_rank"] = r
        rows.append(row)
    res = pd.DataFrame(rows)
    summary = pd.DataFrame({
        name: {"Recall@5": res[f"{name}_recall5"].mean(), "MRR@10": res[f"{name}_rr"].mean()}
        for name in METHODS
    }).T.round(2)
    return res, summary

res, summary = evaluate()
print("Recall@5 = share of the correct coaches that appear in the top 5")
print("MRR@10   = 1 / position of the first correct coach (0 if not in the top 10), averaged\n")
print(summary)

Recall@5 = share of the correct coaches that appear in the top 5
MRR@10   = 1 / position of the first correct coach (0 if not in the top 10), averaged

         Recall@5  MRR@10
keyword      0.80    0.73
vector       0.97    0.88
hybrid       0.97    0.87


In [ ]:
# 9. Per-query view: where is the first correct coach? ("-" = not in the top 10)
fmt = lambda r: "-" if r is None or (isinstance(r, float) and np.isnan(r)) else str(int(r))
per_query = pd.DataFrame({
    "query": res["query"],
    **{name: res[f"{name}_rank"].map(fmt) for name in METHODS},
})
per_query

,query,keyword,vector,hybrid
0,Konfliktmanagement für unser Vertriebsteam,-,1,2
1,Scrum Einführung,1,1,1
2,DISG Persönlichkeitstest,1,1,1
3,Teamcoaching für Remote-Teams,1,1,1
4,Mitarbeiter haben Angst vor der neuen Software,2,1,1
5,"Wie bleibe ich ruhig, wenn ein Kunde schreit?",1,1,1
6,Burnout Prävention,1,1,1
7,sales training cold calling,-,2,3
8,Präsentationstraining,1,1,1
9,neue Führungskraft: vom Kollegen zum Chef,1,1,1


In [ ]:
# 10. Error analysis: look at every query where at least one method missed the top 3
for g, (_, row) in zip(GOLDEN, res.iterrows()):
    ranks = {name: row[f"{name}_rank"] for name in METHODS}
    if all(r is not None and not np.isnan(r) and r <= 3 for r in ranks.values()):
        continue
    print(f"\nQUERY: {g['query']}  {g.get('filters', '')}   (tests: {g['tests']})")
    print("  correct:", ", ".join(f"{i} {by_id[i]['title']}" for i in g["relevant"]))
    for name, fn in METHODS.items():
        top3 = fn(g["query"], g.get("filters", {}))[:3]
        print(f"  {name:<8} rank {fmt(ranks[name]):>2} | top 3:",
              " · ".join(by_id[i]["title"][:30] for i, _ in top3) or "(nothing found)")


QUERY: Konfliktmanagement für unser Vertriebsteam     (tests: synonym: no profile says 'Konfliktmanagement')
  correct: c01 Konfliktcoach für Teams, c25 Mediation bei Konflikten
  keyword  rank  - | top 3: Präsentationstraining für den  · Teamcoaching für verteilte Tea · Karriere- und Laufbahncoaching
  vector   rank  1 | top 3: Mediation bei Konflikten · Konfliktcoach für Teams · Kundenservice und Beschwerdema
  hybrid   rank  2 | top 3: Teamcoaching für verteilte Tea · Konfliktcoach für Teams · Führungskräftetraining

QUERY: sales training cold calling     (tests: English query, German profiles)
  correct: c10 Vertriebstraining
  keyword  rank  - | top 3: Präsentationstraining für den  · DISG-Training
  vector   rank  2 | top 3: Präsentationstraining für den  · Vertriebstraining · Verhandlungstraining
  hybrid   rank  3 | top 3: Präsentationstraining für den  · DISG-Training · Vertriebstraining

QUERY: Gehalt verhandeln     (tests: German compound word + distractor (c06))
  correct:

In [ ]:
# 11. Copy-paste tables for the README
print("| Method | Recall@5 | MRR@10 |\n|---|---|---|")
for name, r in summary.iterrows():
    print(f"| {name} | {r['Recall@5']:.2f} | {r['MRR@10']:.2f} |")
print("\n| Query | keyword | vector | hybrid |\n|---|---|---|---|")
for _, r in per_query.iterrows():
    print(f"| {r['query']} | {r['keyword']} | {r['vector']} | {r['hybrid']} |")

| Method | Recall@5 | MRR@10 |
|---|---|---|
| keyword | 0.80 | 0.73 |
| vector | 0.97 | 0.88 |
| hybrid | 0.97 | 0.87 |

| Query | keyword | vector | hybrid |
|---|---|---|---|
| Konfliktmanagement für unser Vertriebsteam | - | 1 | 2 |
| Scrum Einführung | 1 | 1 | 1 |
| DISG Persönlichkeitstest | 1 | 1 | 1 |
| Teamcoaching für Remote-Teams | 1 | 1 | 1 |
| Mitarbeiter haben Angst vor der neuen Software | 2 | 1 | 1 |
| Wie bleibe ich ruhig, wenn ein Kunde schreit? | 1 | 1 | 1 |
| Burnout Prävention | 1 | 1 | 1 |
| sales training cold calling | - | 2 | 3 |
| Präsentationstraining | 1 | 1 | 1 |
| neue Führungskraft: vom Kollegen zum Chef | 1 | 1 | 1 |
| ChatGPT Workshop für Mitarbeitende | 2 | 1 | 1 |
| Gehalt verhandeln | - | 5 | 5 |
| Einarbeitung neuer Kollegen im Homeoffice | 1 | 2 | 1 |
| Datenschutz Schulung für das ganze Team | 1 | 1 | 1 |
| Strategieworkshop moderieren | 1 | 1 | 1 |


In [ ]:
# 12. Experiment: found a failure -> fix it -> measure again
# Error analysis showed BM25 matching filler words like "für" and "unser", and that noise can
# pull the hybrid ranking down. Fix: remove common German (and English) filler words, then re-run the SAME eval.
before = summary.copy()
before_ranks = per_query.copy()

STOPWORDS = {"für", "und", "der", "die", "das", "den", "dem", "des", "ein", "eine", "einen", "einem", "einer",
             "unser", "unsere", "unseren", "mit", "im", "in", "auf", "zu", "zum", "zur", "von", "vom", "bei",
             "als", "an", "am", "wie", "ich", "wir", "es", "ist", "sind", "haben", "hat", "wenn", "nicht",
             "oder", "aber", "auch", "sich", "über", "ganze", "the", "a", "for", "and", "of"}
bm25 = BM25Okapi([tokenize(doc_text(p)) for p in PROFILES])     # rebuild the index with the new tokenizer

res, summary = evaluate()
per_query = pd.DataFrame({"query": res["query"], **{n: res[f"{n}_rank"].map(fmt) for n in METHODS}})

print("BEFORE (no stop-words removed):\n", before, "\n")
print("AFTER  (stop-words removed):\n", summary, "\n")
changed = before_ranks[(before_ranks[list(METHODS)] != per_query[list(METHODS)]).any(axis=1)]
print("Queries whose ranks changed (before -> after):")
for i in changed.index:
    print(f"  {per_query.loc[i, 'query']:<48}",
          "  ".join(f"{n}: {before_ranks.loc[i, n]} → {per_query.loc[i, n]}" for n in METHODS))
# Note: the vector column can't change here; only the keyword index was rebuilt.

BEFORE (no stop-words removed):
          Recall@5  MRR@10
keyword      0.80    0.73
vector       0.97    0.88
hybrid       0.97    0.87 

AFTER  (stop-words removed):
          Recall@5  MRR@10
keyword      0.73    0.67
vector       0.97    0.88
hybrid       1.00    0.90 

Queries whose ranks changed (before -> after):
  Konfliktmanagement für unser Vertriebsteam       keyword: - → -  vector: 1 → 1  hybrid: 2 → 1
  Wie bleibe ich ruhig, wenn ein Kunde schreit?    keyword: 1 → -  vector: 1 → 1  hybrid: 1 → 1


## What this shows, and what it doesn't

- **Keyword search** is precise for exact terms (Scrum, DISG, ChatGPT), but it fails on synonyms, on English queries and on German compound words ("Gehalt verhandeln" vs. "Gehaltsverhandlung").
- **Semantic search** finds meaning across words and languages, but it can rank a near-topic distractor too high.
- **Hybrid (RRF)** combines both without any training. The question is whether it keeps the strengths of each. Check the table.

**Limits (say these yourself):** 30 profiles and 15 queries is far too small for strong claims; the labels are one person's judgement; relevance is yes/no, not graded.

**Next steps:**
1. A bigger golden set labelled by domain experts, with graded relevance (perfect / good / no) and nDCG.
2. LLM query understanding: free text → JSON (topic, city, budget, language), validated with a schema before filtering.
3. A cross-encoder reranker on the top 50, with the same eval to prove it helps.
4. Online signals: clicks, contact requests, bookings.